### Building a RAG System with LangChain and ChromaDB
#### Introduction

Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [1]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



In [50]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
import tempfile
import os

In [11]:
# Store it in your current working directory instead of /tmp
temppath = tempfile.mkdtemp(suffix="data", dir=".")
print(temppath)

c:\Users\khem6\Desktop\Training\rag-bootcamp-projects\3_Vector_Store_and_DBs\tmp3pewm1n4data


### Sample Data

In [12]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


['\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    ',
 '\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial neural networks. \n    These networks are inspired by the human brain and consist of layers of interconnected \n    nodes. Deep learning has revolutionized fields like computer vision, natural language \n    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly \n    effective f

In [18]:
for i, doc in enumerate(sample_docs):
    with open(f"{temppath}/doc_{i}.txt", "w", encoding="utf-8") as f:
        f.write(doc)

### 1. Document Loading: Load documents from various sources

In [ ]:
dir_loader = DirectoryLoader(
    temppath, 
    glob="*.txt", 
    loader_cls=TextLoader, 
    loader_kwargs={'encoding': 'utf-8'} 
)

dir_loader

In [35]:
documents = dir_loader.load()
documents

[Document(metadata={'source': 'c:\\Users\\khem6\\Desktop\\Training\\rag-bootcamp-projects\\3_Vector_Store_and_DBs\\tmp3pewm1n4data\\doc_0.txt'}, page_content='\n    Machine Learning Fundamentals\n\n    Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through \n    interaction with an environment using rewards and penalties.\n    '),
 Document(metadata={'source': 'c:\\Users\\khem6\\Desktop\\Training\\rag-bootcamp-projects\\3_Vector_Store_and_DBs\\tmp3pewm1n4data\\doc_1.txt'}, page_content='\n    Deep Learning and Neural Networks\n\n    Deep learning is a subset of machine learning based on artificial 

### 2. Document Splitting: Break documents into smaller chunks

In [48]:
r_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""]
)
r_splitter

In [52]:
doc_chunks = r_splitter.split_documents(documents)
doc_chunks

[Document(metadata={'source': 'c:\\Users\\khem6\\Desktop\\Training\\rag-bootcamp-projects\\3_Vector_Store_and_DBs\\tmp3pewm1n4data\\doc_0.txt'}, page_content='Machine Learning Fundamentals'),
 Document(metadata={'source': 'c:\\Users\\khem6\\Desktop\\Training\\rag-bootcamp-projects\\3_Vector_Store_and_DBs\\tmp3pewm1n4data\\doc_0.txt'}, page_content='Machine learning is a subset of artificial intelligence that enables systems to learn \n    and improve from experience without being explicitly programmed. There are three main \n    types of machine learning: supervised learning, unsupervised learning, and reinforcement \n    learning. Supervised learning uses labeled data to train models, while unsupervised \n    learning finds patterns in unlabeled data. Reinforcement learning learns through'),
 Document(metadata={'source': 'c:\\Users\\khem6\\Desktop\\Training\\rag-bootcamp-projects\\3_Vector_Store_and_DBs\\tmp3pewm1n4data\\doc_0.txt'}, page_content='interaction with an environment using

### 3. Embedding Generation: Convert chunks into vector representations

In [51]:
hf_embeddings = HuggingFaceEmbeddings()
hf_embeddings

c:\Users\khem6\Desktop\Training\rag-bootcamp-projects\.venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\khem6\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6553.19it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [60]:
query_vectors = hf_embeddings.embed_query("What is Love?")
query_vectors

[0.05899183452129364,
 -0.0385877750813961,
 0.04229960963129997,
 -0.031247273087501526,
 0.015881549566984177,
 0.041317813098430634,
 -0.1177884191274643,
 0.03236478194594383,
 0.022942468523979187,
 -0.02429167553782463,
 0.012764403596520424,
 -0.04392886161804199,
 0.02628697082400322,
 -0.00829281471669674,
 -0.01072979997843504,
 -0.028157291933894157,
 0.025970617309212685,
 0.024621307849884033,
 0.00023774258443154395,
 0.016626406461000443,
 0.00546799274161458,
 0.00020270945969969034,
 0.028235655277967453,
 -0.038130808621644974,
 0.0044652363285422325,
 -0.0001840724580688402,
 -0.020823918282985687,
 -0.005335732828825712,
 -0.05591483414173126,
 0.03700052201747894,
 -0.01575152389705181,
 -0.054264020174741745,
 -0.006292182952165604,
 0.013620119541883469,
 1.5138806475079036e-06,
 -0.06440024077892303,
 0.011518586426973343,
 -0.001895738416351378,
 0.02531116083264351,
 -0.03852742910385132,
 0.006839025299996138,
 -0.02773248590528965,
 0.052160922437906265,
 0.

In [65]:
texts = [doc.page_content for doc in doc_chunks]
texts_embeddings = hf_embeddings.embed_documents(texts)
print(f"Vectors length: {len(texts_embeddings)}")
print(f"Vectors: {(texts_embeddings)}")


Vectors length: 7
Vectors: [[-0.0213206447660923, -0.030977502465248108, -0.0634642019867897, -0.011823897249996662, -0.0015141115291044116, 0.03522346168756485, 0.027507921680808067, -0.008452067151665688, -0.00957339908927679, 0.017727402970194817, 0.061722200363874435, 0.005039514508098364, 0.022714214399456978, 0.06385095417499542, 0.02172272466123104, -0.09938062727451324, -0.01593439094722271, -0.017801998183131218, 0.02576635219156742, -0.050456803292036057, -0.011339418590068817, -0.04385850951075554, 0.020843109115958214, -0.00517682870849967, -0.06728863716125488, 0.0221165269613266, 0.007692815735936165, -0.0009840134298428893, -0.0579068697988987, 0.00493166409432888, -0.0004449044936336577, -0.054074957966804504, -0.015557841397821903, 0.04363079369068146, 1.053448045240657e-06, -0.0518726110458374, 0.010895985178649426, 0.007355986163020134, -0.016381682828068733, -0.05728968232870102, 0.0033723751548677683, 0.008815811946988106, 0.014883474446833134, 0.001191203482449054

In [72]:
# Simulate storage
import pandas as pd

assert len(texts_embeddings) == len(doc_chunks), "Mismatched lengths!"

storage = []
for i, vec in enumerate(texts_embeddings):
    storage.append({
        "vector": vec,
        "document": doc_chunks[i]
    })
    
storage_df = pd.DataFrame([
    {"vector": s["vector"], "text": s["document"].page_content, "metadata": s["document"].metadata}
    for s in storage
])

storage_df

,vector,text,metadata
0,"[-0.0213206447660923, -0.030977502465248108, -...",Machine Learning Fundamentals,{'source': 'c:\Users\khem6\Desktop\Training\ra...
1,"[-0.0018844736041501164, -0.02440994419157505,...",Machine learning is a subset of artificial int...,{'source': 'c:\Users\khem6\Desktop\Training\ra...
2,"[0.04554951190948486, 0.022053398191928864, -0...",interaction with an environment using rewards ...,{'source': 'c:\Users\khem6\Desktop\Training\ra...
3,"[-0.036853257566690445, 0.07151111215353012, -...",Deep Learning and Neural Networks,{'source': 'c:\Users\khem6\Desktop\Training\ra...
4,"[-0.0112040089443326, -0.0016444639768451452, ...",Deep learning is a subset of machine learning ...,{'source': 'c:\Users\khem6\Desktop\Training\ra...
5,"[-0.06252295523881912, 0.06382513791322708, -0...",excel at sequential data processing.,{'source': 'c:\Users\khem6\Desktop\Training\ra...
6,"[0.06457481533288956, -0.006244209595024586, -...",Natural Language Processing (NLP)\n\n NLP i...,{'source': 'c:\Users\khem6\Desktop\Training\ra...


### 4. Vector Storage: Store embeddings in ChromaDB

### 5. Query Processing: Convert user query to embedding

### 6. Similarity Search: Find relevant chunks from vector store

### 7. Context Augmentation: Combine retrieved chunks with query

### 8. Response Generation: LLM generates answer using context